<a href="https://colab.research.google.com/github/mr-nishanth/-gaitSC/blob/master/IncomeTax_FAQ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Documents Loader

In [1]:
!pip install langchain-community --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
!pip install pypdf --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 5.6 MB/s eta 0:00:00


In [3]:
from langchain.document_loaders import PyPDFLoader

In [4]:
loader = PyPDFLoader("/content/faqs-budget-2025.pdf")

In [5]:
pdf_pages = loader.load()

In [6]:
pdf_pages

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-02-01T14:49:44+05:30', 'author': '', 'moddate': '2025-02-01T14:49:44+05:30', 'title': 'Microsoft Word - FAQs- Budget (1)', 'source': '/content/faqs-budget-2025.pdf', 'total_pages': 61, 'page': 0, 'page_label': '1'}, page_content='FAQ.1: Personal Income-tax reforms with special focus on middle class \nQ.1. What is ‘New Regime’? \nAns. New regime provides for concessional tax rates and liberal slabs. However, no \ndeductions are allowed in the new regime (other than those specified for e.g. 80JJAA, 80M, \nstandard deduction). \n \nQ.2. What are the tax slabs in earlier new regime? \nAns. The Finance (No.2) Act, 2024 had the following slabs in the new tax regime for person, \nbeing an individual or Hindu undivided family or associaƟon of persons [other than a co-\noperaƟve society], or body of individuals, whether incorporated or not, or an arƟﬁcial juridical \nperson referred to in sub-cl

In [10]:
len(pdf_pages)

61

# 2. Documenet Chucking

In [7]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [8]:
chunk_size = 1024
chunk_overlap = 200

In [9]:
splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)

In [11]:
split_docs = splitter.split_documents(pdf_pages)

In [13]:
len(split_docs)

124

# 3: Store In Vector Storage


In [14]:
!pip install langchain-huggingface --quiet

In [15]:
from langchain_huggingface import HuggingFaceEmbeddings

In [16]:
multilingual_embeddings = HuggingFaceEmbeddings(model="intfloat/multilingual-e5-large")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [17]:
sen1 = "I live Dhoni"

In [34]:
sen2 = "I Live Kohli"

In [19]:
sen3 = "Tesla Launched Groq"

In [21]:
test = multilingual_embeddings.embed_query(sen1)

In [22]:
len(test)

1024

In [35]:
emb1 = multilingual_embeddings.embed_query(sen1)
emb2 = multilingual_embeddings.embed_query(sen2)
emb3 = multilingual_embeddings.embed_query(sen3)

In [30]:
import numpy as np

In [36]:
np.dot(emb1, emb2)

np.float64(0.9194800301873789)

In [37]:
np.dot(emb1, emb3)

np.float64(0.7729119125601216)

In [38]:
np.dot(emb2, emb3)

np.float64(0.7855130590874972)

In [39]:
!pip install chromadb --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 1.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.8/510.8 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 55.8 MB/s eta 0:0

In [40]:
from langchain.vectorstores import Chroma

In [42]:
chroma_instance = Chroma.from_documents(split_docs, multilingual_embeddings, persist_directory="db")

In [43]:
!pip install langchain-groq --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 3.1 MB/s eta 0:00:00


In [46]:
import getpass

In [47]:
import os

In [48]:
os.environ["GROQ_API_KEY"] = getpass.getpass("Enter Your GROQ API Key")

Enter Your GROQ API Key··········


In [49]:
from langchain_groq import ChatGroq

In [51]:
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_tokens=512)

In [52]:
system_prompt = (
        "You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question."
        "If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise."
        "Answer all questions to the best of your ability."
    )

In [53]:
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage

In [54]:
system_message = SystemMessage(content=system_prompt)

In [55]:
system_message

SystemMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question.If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.Answer all questions to the best of your ability.", additional_kwargs={}, response_metadata={})

In [77]:
question = " What is the total income till which marginal relief is admissible? "

In [78]:
docs = chroma_instance.similarity_search_with_score(query=question,k=5)

In [66]:
docs

[(Document(metadata={'creationdate': '2025-02-01T14:49:44+05:30', 'author': '', 'producer': 'Microsoft: Print To PDF', 'title': 'Microsoft Word - FAQs- Budget (1)', 'page_label': '6', 'page': 5, 'creator': 'PyPDF', 'source': '/content/faqs-budget-2025.pdf', 'moddate': '2025-02-01T14:49:44+05:30', 'total_pages': 61}, page_content='Ans. No rebate is not available on income from capital gains or loƩeries or any other income \non which special rate has been provided in the Act. It is available only on the tax payable as \nper slabs under secƟon 115BAC. \n \nQ.21. What is the diﬀerence between rebate and marginal relief? \nAns: Rebate is the deducƟon from tax which is available to tax payers having income upto Rs. \n12 Lacs in the new regime. Marginal relief ensures that taxpayers having income marginally \nhigher than Rs. 12 lacs do not pay tax more than the income in excess of 12 lacs.'),
  0.2106817066669464),
 (Document(metadata={'total_pages': 61, 'source': '/content/faqs-budget-2025.p

In [79]:
import pandas as pd

_docs = pd.DataFrame(
                    [(question, doc[0].page_content, doc[0].metadata.get('source'), doc[0].metadata.get('page'), doc[1]) for doc in docs],
                    columns=['query', 'paragraph', 'document', 'page_number', 'relevant_score']
                )

In [70]:
_docs

,query,paragraph,document,page_number,relevant_score
0,What is the difference between rebate and marg...,Ans. No rebate is not available on income from...,/content/faqs-budget-2025.pdf,5,0.210682
1,What is the difference between rebate and marg...,(iv) The marginal relief shall be computed by ...,/content/faqs-budget-2025.pdf,4,0.277698
2,What is the difference between rebate and marg...,"Ans. Presently, for AY 2024-25, about 8.75 cro...",/content/faqs-budget-2025.pdf,3,0.325411
3,What is the difference between rebate and marg...,"Rs 12,70,000 70,500 70,000 \nRs 12,75,000 71,2...",/content/faqs-budget-2025.pdf,3,0.349963
4,What is the difference between rebate and marg...,"relief, the amount of tax to be actually paid ...",/content/faqs-budget-2025.pdf,3,0.358768


In [80]:
context = "\n\n".join(_docs['paragraph'])

In [81]:
context

'(iv) The marginal relief shall be computed by deducting the income exceeding Rs. 12, \n10,000 (i.e. Rs.10,000) from total tax liability determined in this case (i.e. Rs. \n61,500) as tabulated above. \n(v) Therefore, in the above case rebate by way of marginal relief is Rs. 51,500 \n(61,500/- 10,000/-= 51,500/-) is allowed. \n(vi) Tax payable is therefore Rs. 10,000 [Rs. 61,500-Rs.51,500] \n  \nQ.18. What is the maximum amount of rebate available to any tax payer? \nAns. The maximum rebate available is Rs 60,000 which is there for a tax payer having income \nof Rs 12 lacs on which tax is payable as per the new slabs. \n \nQ. 19. What is the total income till which marginal relief is admissible? \nAns. The total income till which marginal relief is available is near about Rs. 12,75,000/-. \n \nQ.20.   Whether special income having special rate such as capital gains, loƩery etc. also be \neligible for rebate?\n\nAns. Presently, for AY 2024-25, about 8.75 crore persons have filed their I

In [82]:
user_message = HumanMessage(content=f"Context: {context}\n\nQuestion: {question}")

In [83]:
result = llm.invoke([system_message,user_message])

In [75]:
result

AIMessage(content='Rebate is a deduction from tax available to taxpayers having income up to Rs. 12 lacs in the new regime, and it is a maximum of Rs. 60,000. Marginal relief ensures that taxpayers having income marginally higher than Rs. 12 lacs do not pay tax more than the income in excess of 12 lacs. It is computed by deducting the income exceeding Rs. 12 lacs minus Rs. 10,000 from the total tax liability.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 101, 'prompt_tokens': 1223, 'total_tokens': 1324, 'completion_time': 0.12715401, 'prompt_time': 0.067528768, 'queue_time': 0.187842602, 'total_time': 0.194682778}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_a7a2f9abbf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--f92a7521-d7c4-49d4-9be8-d69bacf5997a-0', usage_metadata={'input_tokens': 1223, 'output_tokens': 101, 'total_tokens': 1324})

In [84]:
result.content

'The total income till which marginal relief is available is approximately Rs. 12,75,000.'